# 05 — Aggressive Quantization Tradeoff: Does Pushing Bits Down Close the Memory Gap?

**Question this answers**: notebook 04 found the flagship SGT-QAT recipe's
compressed checkpoint (avg 3.156 bits/weight, mixed W4/W3) has a standalone VRAM
footprint of 1.629 GiB — already bigger than EAGLE-3's full in-vLLM memory delta
(1.047 GiB). Back-of-envelope math suggested an average of ~2 bits/weight might
close that gap. **This notebook actually tests that**, instead of leaving it as
speculation.

**Method**: same recipe as `notebooks/01_export_sgt_qat_checkpoint.ipynb`
(sensitivity-ranked mixed-precision GPTQ + targeted QAT on the lower-bit layers),
shifted down one tier: protected layers get **W3** (was W4), the rest get **W2**
(was W3) — average ~2.15 bits/weight at the flagship `PROTECT_FRAC=0.15`. Same
seed (42), same hyperparameters, for direct comparability against notebook 01's
numbers.

**Deliberately cheap**: no 8B target model, no vLLM, no full acceptance-rate
benchmark here — this only needs the 1.7B model (same order of cost as notebook
01, which ran fine on a single Colab session). Quality is assessed via WikiText-2
PPL (a proxy, comparable to notebook 01's own PPL numbers) rather than a full
in-vLLM acceptance-rate re-run — that's a separate, more expensive follow-up if
this notebook's PPL looks survivable.

**Known risk**: 2-bit quantization is genuinely aggressive. This may produce a
badly degraded model (high PPL, possibly NaN loss during QAT) -- that outcome is
itself the answer to the question ("no, can't push this far without breaking
quality"), not a bug to fix.

**Not yet run.**

## Setup

No vLLM needed in this notebook (no generation/serving, just quantization +
training + a standalone memory read) -- skips the whole libcudart/CUDA-13
class of issues that plagued notebooks 02/03.

In [ ]:
import os
REPO_NAME = 'sgt-qat-draft'
if not os.path.isdir(REPO_NAME):
    # Anonymous clones from Colab's shared IPs can hit GitHub auth/rate-limit
    # failures (docs/logs.md 2026-07-25) -- clone with the GITHUB_TOKEN Colab
    # secret by default so this can't recur.
    from google.colab import userdata
    _token = userdata.get('GITHUB_TOKEN')
    _clone_url = f'https://{_token}@github.com/Resh19S/sgt-qat-draft.git'
    !git clone {_clone_url}
%cd {REPO_NAME}
!git pull

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!pip install -q llmcompressor bitsandbytes compressed-tensors
!pip uninstall -y scikit-learn scipy torchao -q

import sys, torch, json, math, gc, subprocess
from pathlib import Path
from datetime import datetime, timezone
import torch.nn as nn
from datasets import load_dataset, Dataset

print(torch.__version__)
assert torch.cuda.is_available(), "No GPU detected."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
MODEL_ID = 'Qwen/Qwen3-1.7B'
CALIB_SEED = 42          # same as notebook 01, for direct comparability
SEQ_LEN = 2048
CALIB_N = 128
PROTECT_FRAC = 0.15      # same as notebook 01

# The one deliberate change from notebook 01: shifted down one bit-width tier.
PROTECTED_BITS = 3       # was 4 in notebook 01
REST_BITS = 2            # was 3 in notebook 01

REPO_DIR = Path('.').resolve()
CHECKPOINT_DIR = REPO_DIR / 'checkpoints' / 'qwen3-1.7b-sgt-qat-aggressive'
RESULTS_DIR = REPO_DIR / 'results'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Reference points to compare against.
NOTEBOOK_01_REFERENCE = {
    'avg_bits_per_weight': 0.1562 * 4 + 0.8438 * 3,  # 3.156
    'ppl_stage1_only': 22.37,
    'ppl_combined': 15.91,
    'compressed_standalone_memory_gib': 1.629,  # from notebook 04
}
EAGLE3_REFERENCE_MEMORY_GIB = 1.047  # full in-vLLM delta, from notebook 02's matched-session run
print(f"Target average bits/weight this run: {0.1562*PROTECTED_BITS + 0.8438*REST_BITS:.3f} "
      f"(notebook 01 was {NOTEBOOK_01_REFERENCE['avg_bits_per_weight']:.3f})")

In [ ]:
import transformers.tokenization_utils_base as _tub
if hasattr(_tub, 'list_repo_templates'):
    _original_list_repo_templates = _tub.list_repo_templates
    def _safe_list_repo_templates(*args, **kwargs):
        try:
            return _original_list_repo_templates(*args, **kwargs)
        except Exception:
            return []
    _tub.list_repo_templates = _safe_list_repo_templates

from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)


def get_calibration_data(tokenizer, n=CALIB_N, seq_len=SEQ_LEN, seed=CALIB_SEED):
    ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=10_000)
    samples, collected = [], 0
    for item in ds:
        enc = tokenizer(item['text'], return_tensors='pt', truncation=True, max_length=seq_len)
        if enc['input_ids'].shape[1] == seq_len:
            samples.append(enc['input_ids'])
            collected += 1
            if collected >= n:
                break
    return torch.cat(samples, dim=0)


def compute_perplexity_wikitext2(model, tokenizer, seq_len=SEQ_LEN, stride=2048):
    ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n\n'.join(ds['text'])
    enc = tokenizer(text, return_tensors='pt')
    input_ids = enc['input_ids'].to(model.device)
    seq_len_total = input_ids.size(1)
    nlls, prev_end = [], 0
    model.eval()
    for begin in range(0, seq_len_total - seq_len, stride):
        end = begin + seq_len
        target_len = end - max(begin, prev_end)
        with torch.no_grad():
            out = model(input_ids[:, begin:end], labels=input_ids[:, begin:end])
        nlls.append(out.loss * target_len)
        prev_end = end
    return float(math.exp(torch.stack(nlls).sum() / prev_end))

## 1. Per-layer sensitivity ranking

Same method as notebook 01. Recomputed fresh (no cache) -- sensitivity in
principle could shift slightly at different candidate bit-widths, but reusing
the same W3-vs-W4 probe here (matching notebook 01 exactly) keeps this directly
comparable rather than introducing a second variable.

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier


def build_gptq_checkpoint(model_id, tokenizer, bits, seed=CALIB_SEED):
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)
    calib_tensor = get_calibration_data(tokenizer, seed=seed)
    calib_dataset = Dataset.from_dict({'input_ids': calib_tensor.tolist()})
    recipe = GPTQModifier(dampening_frac=0.01, ignore=['lm_head'],
        config_groups={'group_0': {'targets': ['Linear'], 'input_activations': None, 'output_activations': None,
                                    'weights': {'num_bits': bits, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128}}})
    oneshot(model=model, dataset=calib_dataset, recipe=recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)
    return model


def build_probe_ids(tokenizer, seq_len=SEQ_LEN, n_windows=4):
    ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n\n'.join(ds['text'])
    enc = tokenizer(text, return_tensors='pt')
    return enc['input_ids'][:, :n_windows * seq_len]


def compute_perplexity_probe(model, probe_ids, seq_len=SEQ_LEN):
    ids = probe_ids.to(model.device)
    total_len = ids.size(1)
    nlls, prev_end = [], 0
    model.eval()
    for begin in range(0, total_len - seq_len + 1, seq_len):
        end = begin + seq_len
        target_len = end - max(begin, prev_end)
        with torch.no_grad():
            out = model(ids[:, begin:end], labels=ids[:, begin:end])
        nlls.append(out.loss * target_len)
        prev_end = end
    return float(math.exp(torch.stack(nlls).sum() / prev_end))


def get_module(model, name):
    mod = model
    for part in name.split('.'):
        mod = getattr(mod, part)
    return mod


print("Building throwaway W3/W4 checkpoints to derive the sensitivity ranking (same probe as notebook 01)...")
model_w3_tmp = build_gptq_checkpoint(MODEL_ID, tokenizer, bits=3, seed=CALIB_SEED)
model_w4_tmp = build_gptq_checkpoint(MODEL_ID, tokenizer, bits=4, seed=CALIB_SEED)
probe_ids = build_probe_ids(tokenizer)
ppl_w3_probe = compute_perplexity_probe(model_w3_tmp, probe_ids)

layer_names_tmp = [name for name, module in model_w3_tmp.named_modules()
                   if isinstance(module, nn.Linear) and hasattr(module, 'weight_scale')]
sensitivity = []
for name in layer_names_tmp:
    dst = get_module(model_w3_tmp, name)
    src = get_module(model_w4_tmp, name)
    snap = dst.weight.data.clone()
    dst.weight.data.copy_(src.weight.data)
    hybrid_ppl = compute_perplexity_probe(model_w3_tmp, probe_ids)
    dst.weight.data.copy_(snap)
    n_params = dst.weight.numel()
    improvement = ppl_w3_probe - hybrid_ppl
    sensitivity.append({'layer': name, 'n_params': n_params,
                        'value_per_mparam': improvement / (n_params / 1e6) if n_params else 0})
sensitivity.sort(key=lambda d: d['value_per_mparam'], reverse=True)
del model_w3_tmp, model_w4_tmp
gc.collect(); torch.cuda.empty_cache()
print(f"Computed sensitivity ranking for {len(sensitivity)} layers.")

## 2. Stage 1 — mixed-precision GPTQ at the aggressive bit-widths (W3 protected / W2 rest)

In [ ]:
ranked = sorted(sensitivity, key=lambda d: d['value_per_mparam'], reverse=True)
total_params = sum(d['n_params'] for d in ranked)
target_params = PROTECT_FRAC * total_params

protected_layers, cumulative_params = [], 0
for d in ranked:
    if cumulative_params >= target_params:
        break
    protected_layers.append(d['layer'])
    cumulative_params += d['n_params']

actual_frac = cumulative_params / total_params
print(f"Protecting {len(protected_layers)} layers at W{PROTECTED_BITS} ({actual_frac*100:.1f}% of quantized params)")

_probe_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cpu', trust_remote_code=True)
all_linear_names = [name for name, module in _probe_model.named_modules()
                    if isinstance(module, nn.Linear) and name != 'lm_head']
del _probe_model
unprotected_layers = [n for n in all_linear_names if n not in set(protected_layers)]
print(f"{len(protected_layers)} layers -> W{PROTECTED_BITS}, {len(unprotected_layers)} layers -> W{REST_BITS} (of {len(all_linear_names)} total)")

model_mixed = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)
calib_tensor = get_calibration_data(tokenizer, seed=CALIB_SEED)
calib_dataset = Dataset.from_dict({'input_ids': calib_tensor.tolist()})

mixed_recipe = GPTQModifier(
    dampening_frac=0.01, ignore=['lm_head'],
    config_groups={
        'protected_group': {
            'targets': protected_layers,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': PROTECTED_BITS, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
        'rest_group': {
            'targets': unprotected_layers,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': REST_BITS, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
    },
)
oneshot(model=model_mixed, dataset=calib_dataset, recipe=mixed_recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

ppl_mixed_stage1 = compute_perplexity_wikitext2(model_mixed, tokenizer)
print(f"Stage 1 (mixed precision alone) PPL: {ppl_mixed_stage1:.2f}")
print(f"Notebook 01's Stage 1 PPL for comparison: {NOTEBOOK_01_REFERENCE['ppl_stage1_only']:.2f} (at W4/W3, not W3/W2)")

## 3. Stage 2 — targeted QAT on the still-low-bit (W2) layers

Same hyperparameters as notebook 01 for direct comparability. If loss goes
non-finite, the training loop raises immediately (same safety net as notebook
01) -- that itself would be informative (2-bit training instability), not
something to silently work around.

In [ ]:
import bitsandbytes as bnb

class FakeQuantize(nn.Module):
    def __init__(self, bits):
        super().__init__()
        self.qmin = -(2 ** (bits - 1))
        self.qmax = (2 ** (bits - 1)) - 1

    def forward(self, x, scale, zero_point):
        out_features, in_features = x.shape
        num_groups = scale.shape[-1]
        group_size = in_features // num_groups
        x_grouped = x.reshape(out_features, num_groups, group_size)
        scale_b = scale.unsqueeze(-1).to(x.dtype)
        zp_b = zero_point.unsqueeze(-1).to(x.dtype)
        x_q = ((x_grouped / scale_b) + zp_b).round().clamp(self.qmin, self.qmax)
        x_dq = (x_q - zp_b) * scale_b
        x_ste = x_grouped + (x_dq - x_grouped).detach()
        return x_ste.reshape(out_features, in_features)

def insert_fake_quant_on_subset(model, layer_names, bits):
    handles = []
    fq = FakeQuantize(bits=bits)
    target_set = set(layer_names)
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and hasattr(module, 'weight_scale') and name in target_set:
            def make_hook(m):
                def hook(mod, inp):
                    mod.weight.data = fq(mod.weight.data, mod.weight_scale, mod.weight_zero_point)
                return hook
            handles.append(module.register_forward_pre_hook(make_hook(module)))
    return handles

LR = 1e-5
TRAIN_STEPS = 500
BATCH_TOKENS = 1024  # matches notebook 01's validated A100 run

ds_train = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
train_text = '\n\n'.join(ds_train['text'])
all_ids = tokenizer(train_text, return_tensors='pt')['input_ids'][0].to('cuda')

QAT_PRECISION = 'fp32'
model_mixed = model_mixed.float()
gc.collect(); torch.cuda.empty_cache()

fq_handles = insert_fake_quant_on_subset(model_mixed, unprotected_layers, bits=REST_BITS)
n_trainable = 0
for name, module in model_mixed.named_modules():
    if isinstance(module, nn.Linear) and hasattr(module, 'weight_scale'):
        module.weight_scale.requires_grad_(False)
        is_still_low_bit = name in set(unprotected_layers)
        module.weight.requires_grad_(is_still_low_bit)
        if is_still_low_bit:
            n_trainable += module.weight.numel()
print(f"Trainable parameters (still-W{REST_BITS} layers only): {n_trainable/1e6:.1f}M")

trainable_params = [p for p in model_mixed.parameters() if p.requires_grad]
optimizer = bnb.optim.AdamW8bit(trainable_params, lr=LR)
model_mixed.train()
if hasattr(model_mixed, 'gradient_checkpointing_enable'):
    model_mixed.gradient_checkpointing_enable()

for step in range(TRAIN_STEPS):
    start = (step * BATCH_TOKENS) % (len(all_ids) - BATCH_TOKENS - 1)
    chunk = all_ids[start:start + BATCH_TOKENS].unsqueeze(0)
    optimizer.zero_grad()
    out = model_mixed(chunk, labels=chunk)
    if not torch.isfinite(out.loss):
        raise RuntimeError(
            f"Loss non-finite at step {step} -- W{REST_BITS} QAT may genuinely be too "
            "aggressive for this training recipe. This is itself an answer to this "
            "notebook's question, not (necessarily) a bug."
        )
    out.loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()
    if step % 100 == 0:
        print(f"  step {step}/{TRAIN_STEPS}  loss={out.loss.item():.4f}")

for h in fq_handles:
    h.remove()
del optimizer
model_mixed.zero_grad(set_to_none=True)
gc.collect(); torch.cuda.empty_cache()

ppl_combined = compute_perplexity_wikitext2(model_mixed, tokenizer)
print(f"\nStage 1+2 (mixed precision + targeted QAT) PPL: {ppl_combined:.2f}")
print(f"Notebook 01's combined PPL for comparison: {NOTEBOOK_01_REFERENCE['ppl_combined']:.2f} (at W4/W3, not W3/W2)")
if ppl_combined > NOTEBOOK_01_REFERENCE['ppl_combined'] * 2:
    print("\n>>> PPL more than doubled vs. the flagship recipe -- likely a real quality ")
    print(">>> collapse at this bit-width, not noise. Worth stopping and reflecting ")
    print(">>> before spending more compute on the export/memory steps below.")

## 4. Compressed export

In [ ]:
model_mixed = model_mixed.half()
model_mixed.save_pretrained(str(CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(CHECKPOINT_DIR))

def _dir_size_bytes(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file())

checkpoint_size_mb = _dir_size_bytes(CHECKPOINT_DIR) / (1024 ** 2)
print(f"Exported checkpoint size: {checkpoint_size_mb:.0f} MB "
      f"(notebook 01's W4/W3 checkpoint was 1184.8 MB -- should be meaningfully smaller here)")

## 5. Standalone VRAM footprint (same method as notebook 04)

Measures this checkpoint's own compressed weight footprint, deliberately without
decompressing -- directly comparable to notebook 04's 1.629 GiB number for the
flagship W4/W3 checkpoint, and to EAGLE-3's 1.047 GiB full in-vLLM delta (with
the same standalone-vs-full-serving caveat noted there).

In [ ]:
del model_mixed
gc.collect()
torch.cuda.empty_cache()

def _gpu_memory_used_mb(device_index: int = 0) -> int:
    out = subprocess.check_output([
        'nvidia-smi', f'--id={device_index}',
        '--query-gpu=memory.used', '--format=csv,noheader,nounits',
    ])
    return int(out.decode().strip().splitlines()[0]) * 1024 * 1024

baseline_memory_bytes = _gpu_memory_used_mb()
model_reload = AutoModelForCausalLM.from_pretrained(
    str(CHECKPOINT_DIR), dtype=torch.float16, trust_remote_code=True, device_map='cuda',
)
loaded_memory_bytes = _gpu_memory_used_mb()
memory_delta_bytes = loaded_memory_bytes - baseline_memory_bytes
memory_delta_gib = memory_delta_bytes / 1024**3
print(f"Standalone VRAM footprint (this aggressive checkpoint): {memory_delta_gib:.3f} GiB")
print(f"Notebook 04's flagship (W4/W3) checkpoint: {NOTEBOOK_01_REFERENCE['compressed_standalone_memory_gib']:.3f} GiB")
print(f"EAGLE-3 full in-vLLM delta (different measurement context, see findings.md caveat): {EAGLE3_REFERENCE_MEMORY_GIB:.3f} GiB")
del model_reload
gc.collect(); torch.cuda.empty_cache()

## 6. Back up to Drive and log results

Learned from notebook 01: back this up immediately, don't leave it only on the
ephemeral Colab disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/sgt-qat-draft-checkpoints')
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
!cp -r {str(CHECKPOINT_DIR)} {str(DRIVE_BACKUP_DIR / CHECKPOINT_DIR.name)}
!du -sh {str(DRIVE_BACKUP_DIR / CHECKPOINT_DIR.name)}

In [ ]:
record = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'experiment': 'aggressive_quant_tradeoff',
    'model': MODEL_ID,
    'calib_seed': CALIB_SEED,
    'protect_frac_target': PROTECT_FRAC,
    'protect_frac_actual': round(actual_frac, 4),
    'protected_bits': PROTECTED_BITS,
    'rest_bits': REST_BITS,
    'avg_bits_per_weight': round(actual_frac * PROTECTED_BITS + (1 - actual_frac) * REST_BITS, 3),
    'n_layers_protected': len(protected_layers),
    'n_layers_rest': len(unprotected_layers),
    'ppl_stage1_only': ppl_mixed_stage1,
    'ppl_combined': ppl_combined,
    'notebook_01_reference': NOTEBOOK_01_REFERENCE,
    'checkpoint_size_mb': round(checkpoint_size_mb, 1),
    'standalone_memory_delta_bytes': memory_delta_bytes,
    'standalone_memory_delta_gib': round(memory_delta_gib, 3),
    'eagle3_reference_memory_gib': EAGLE3_REFERENCE_MEMORY_GIB,
    'n_trainable_params_stage2': n_trainable,
    'qat_steps': TRAIN_STEPS,
    'qat_lr': LR,
    'qat_precision': QAT_PRECISION,
    'gpu': torch.cuda.get_device_name(0),
}

ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%S')
fname = f"aggressive_quant_tradeoff_seed{CALIB_SEED}_{ts}.json"
(RESULTS_DIR / fname).write_text(json.dumps(record, indent=2))
print(f"Saved: results/{fname}")
print("\nTranscribe into docs/findings.md once this looks sane -- note whether PPL degradation")
print("and/or memory reduction actually moved in the directions/magnitudes hypothesized.")